# Comparação entre splits e resoluções

## Objetivo

Compara Dice, tempo e sucesso dos óstios entre splits e resoluções.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, *NOTEBOOK_CWD.parents):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.runtime.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()

### Funções de apresentação
Os helpers abaixo organizam os gráficos específicos desta análise.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from typing import Any, Optional, Sequence


def prepare_subset_plot_df(subset_summary_df: pd.DataFrame) -> pd.DataFrame:
    """Prepara DataFrame de subconjuntos disponível para plotagem."""
    # Filtra apenas linhas marcadas como disponíveis.
    plot_subset_df = subset_summary_df[subset_summary_df["disponivel"]].copy()
    # Traduz nome do subset para rótulo do eixo X.
    plot_subset_df["conjunto"] = plot_subset_df["subset"].map(
        {"train": "Treino", "val": "Validacao", "test": "Teste"}
    )
    return plot_subset_df


def plot_subset_metric_by_resolution(
    subset_summary_df: pd.DataFrame,
    metric_col: str,
    title: str,
    ylabel: str,
    palette: Any,
    ylim: Optional[tuple] = None,
    hline_y: Optional[float] = None,
    hline_kwargs: Optional[dict] = None,
    bar_label_fmt: str = "%.3f",
) -> Any:
    """Plota uma métrica por subconjunto com separação por resolução."""
    # Gera base de plot com subsets válidos.
    plot_subset_df = prepare_subset_plot_df(subset_summary_df)

    # Barra agrupada por subset e resolução.
    fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
    ax = sns.barplot(
        data=plot_subset_df,
        x="conjunto",
        y=metric_col,
        hue="resolucao",
        palette=palette,
    )

    if ylim is not None:
        # Aplica limite fixo quando definido.
        ax.set_ylim(*ylim)

    if hline_y is not None:
        # Linha de referência opcional.
        line_kwargs = {"color": "black", "linewidth": 1, "linestyle": "--"}
        if hline_kwargs:
            line_kwargs.update(hline_kwargs)
        ax.axhline(hline_y, **line_kwargs)

    ax.set_title(title, pad=16)
    ax.set_xlabel("Conjunto")
    ax.set_ylabel(ylabel, labelpad=10)
    ax.grid(axis="y", alpha=0.3)
    # Legenda identifica cada resolução.
    ax.legend(title="Resolucao")

    for container in ax.containers:
        # Rótulo numérico em cada barra.
        ax.bar_label(container, fmt=bar_label_fmt, padding=3)

    plt.show()
    return ax


def plot_subset_execution_time_by_resolution(
    subset_summary_df: pd.DataFrame,
    palette: Optional[Sequence[str]] = None,
    ymax_factor: float = 1.12,
) -> Any:
    """Plota tempo de execução por subconjunto e resolução."""
    # Plota tempo (min) por subset usando helper comum.
    if palette is None:
        palette = ["#F58518", "#E45756"]

    plot_subset_df = prepare_subset_plot_df(subset_summary_df)
    max_time = plot_subset_df["tempo_execucao_min"].max()
    if pd.notna(max_time):
        ylim = (0, max_time * ymax_factor)
    else:
        ylim = (0, 1)

    # Ajusta o formato do label para reforçar a leitura dos minutos.
    return plot_subset_metric_by_resolution(
        subset_summary_df=subset_summary_df,
        metric_col="tempo_execucao_min",
        title="Tempo de execução por conjunto e resolução",
        ylabel="Tempo de execução (min)",
        palette=palette,
        ylim=ylim,
        bar_label_fmt="%.1f",
    )


def plot_subset_ostia_success_by_resolution(
    subset_summary_df: pd.DataFrame, palette: Optional[Sequence[str]] = None
) -> Any:
    """Plota sucesso de detecção de óstios por subconjunto e resolução."""
    # Plota sucesso total (%) por subset usando helper comum.
    if palette is None:
        palette = ["#54A24B", "#2E8B57"]

    return plot_subset_metric_by_resolution(
        subset_summary_df=subset_summary_df,
        metric_col="sucesso_total_percent",
        title="Percentual de óstios detectados com sucesso por conjunto e resolução",
        ylabel="Óstios detectados com sucesso (%)",
        palette=palette,
        ylim=(0, 100),
        bar_label_fmt="%.1f%%",
    )

In [ ]:
from utils.project.runtime.notebook_env import get_default_split_paths

import utils.comparison_utils as cmp

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="deep",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 110,
        "savefig.dpi": 300,
    },
)
plt.close("all")

SPLIT_PATHS_BY_RESOLUTION = get_default_split_paths(REPO_ROOT)

VALID_SPLITS = ("train", "val", "test")

## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento de dados

### Dados de train, test e validação

In [ ]:
subset_summary_df = cmp.build_split_resolution_summary(
    SPLIT_PATHS_BY_RESOLUTION,
    valid_splits=VALID_SPLITS,
)
display(subset_summary_df)

missing_rows = subset_summary_df[~subset_summary_df["is_available"]]
if not missing_rows.empty:
    print("Combinations without data:")
    display(missing_rows[["subset", "resolution"]])

## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Resultados por subset: val, train e test

In [ ]:
ax_subset = plot_subset_metric_by_resolution(
    subset_summary_df=subset_summary_df,
    metric_col="dice_medio_correto",
    title="Dice médio (both_correct + both_tolerable) por conjunto e resolução",
    ylabel="Dice médio",
    palette=["#4C78A8", "#F58518"],
    ylim=(0, 1),
    bar_label_fmt="%.3f",
)

In [ ]:
ax_subset_all = plot_subset_metric_by_resolution(
    subset_summary_df=subset_summary_df,
    metric_col="dice_medio_todos",
    title="Dice médio (todos os resultados, incluindo casos ruins) por conjunto e resolução",
    ylabel="Dice médio",
    palette=["#4C78A8", "#F58518"],
    ylim=(0, 1),
    bar_label_fmt="%.3f",
)

### Impacto dos casos ruins no Dice

In [ ]:
subset_summary_df["delta_dice"] = (
    subset_summary_df["dice_medio_todos"] - subset_summary_df["dice_medio_correto"]
)

ax_subset_delta = plot_subset_metric_by_resolution(
    subset_summary_df=subset_summary_df,
    metric_col="delta_dice",
    title="Impacto dos casos ruins no Dice por conjunto e resolução",
    ylabel="Dice médio (todos - corretos)",
    palette=["#9C755F", "#B279A2"],
    hline_y=0,
    bar_label_fmt="%.3f",
)

### Tempo de execução por conjunto

In [ ]:
ax_subset_time = plot_subset_execution_time_by_resolution(
    subset_summary_df=subset_summary_df,
    palette=["#F58518", "#E45756"],
)

### Percentual de óstios detectados com sucesso por conjunto

In [ ]:
ax_subset_success = plot_subset_ostia_success_by_resolution(
    subset_summary_df=subset_summary_df,
    palette=["#54A24B", "#2E8B57"],
)

## Conclusão

A tabela consolidada evidencia as combinações disponíveis e permite comparar Dice, tempo e sucesso dos óstios entre splits e resoluções.